# Step 2: IMPRESSION 임베딩 (bge-small-en-v1.5)
880K 판독문을 GPU로 벡터화. A10G 기준 ~2분.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
BUCKET = "pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an"

# S3에서 reports.jsonl 다운로드
!aws s3 cp s3://{BUCKET}/rag/build/reports.jsonl /tmp/reports.jsonl
!ls -lh /tmp/reports.jsonl

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
import numpy as np
import json, time

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}, GPUs: {torch.cuda.device_count()}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
model.max_seq_length = 512
print("모델 로드 완료")

In [ ]:
# 데이터 로드 + 전처리
records = []
with open("/tmp/reports.jsonl", "r") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

impressions = []
for r in records:
    text = (r.get("impression") or "")[:300].strip()
    if len(text) < 3:
        text = "No impression available"
    impressions.append(text)

print(f"총 {len(impressions):,}건")

In [ ]:
# 임베딩
start = time.time()
embeddings = model.encode(
    impressions,
    batch_size=1024,
    show_progress_bar=True,
    normalize_embeddings=True
)
elapsed = time.time() - start
print(f"\n완료: {embeddings.shape} ({elapsed:.0f}초, {elapsed/60:.1f}분)")

In [ ]:
# 저장
np.save("/tmp/embeddings.npy", embeddings)
print(f"embeddings.npy 저장: {embeddings.shape}")

# 메타데이터
with open("/tmp/metadata.jsonl", "w") as f:
    for record in records:
        meta = {
            "note_id": record["note_id"],
            "subject_id": record["subject_id"],
            "hadm_id": record["hadm_id"],
            "charttime": record["charttime"],
            "impression": record["impression"],
            "findings": record["findings"],
            "indication": record["indication"],
            "examination": record["examination"],
            "comparison": record["comparison"],
        }
        f.write(json.dumps(meta, ensure_ascii=False) + "\n")

print("metadata.jsonl 저장")
!ls -lh /tmp/embeddings.npy /tmp/metadata.jsonl

In [ ]:
# S3 업로드
!aws s3 cp /tmp/embeddings.npy s3://{BUCKET}/rag/build/output/embeddings.npy
!aws s3 cp /tmp/metadata.jsonl s3://{BUCKET}/rag/build/output/metadata.jsonl
print("\nS3 업로드 완료!")
!aws s3 ls s3://{BUCKET}/rag/build/output/